# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AL-NAJAF/MLAssgn01/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/AL-NAJAF/MLAssgn01"
REPO_DIR = "MLAssgn01"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Now in:", os.getcwd())

Now in: /content/MLAssgn01


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My lane as an ML task (type)

My lane (Lane 2: Refresh / Content Opportunity Scoring) maps onto a **Ranking / Scoring** task.
The core question is "which pages should be reviewed first?" — not "is this page declining,
yes/no?" and not "what groups of pages exist?". A reviewer has limited time, so what matters
is the ORDER pages are surfaced in, not a single yes/no cutoff. That's exactly what ranking/
scoring tasks are built for: producing a priority score per page so the top of the list is
where a human should look first.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or proxy

For now, my proxy target is `is_declining_label` (trend_direction == "down"), the same label
used in the starter pipeline. This is a PROXY, not a true future outcome — it's a bucket
calculated from the current window, not something that happened after a decision point. A
stronger version for the actual capstone would be a future-looking label built from the
warehouse data (e.g. "declined over the next 30 days, based on the prior 90 days of
features"), which avoids the weakness of a same-window proxy label.

Important: I will never use `trend_direction` or `trend_pct` as FEATURES, since the label is
directly derived from them — using them as inputs would just teach a model to copy the label
instead of finding real signal (a leakage trap I already saw in the Week 2 starter notebook,
where using `trend_pct` as a feature gave a fake Precision@50 of 1.000).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success metric

My success metric is **Precision@50**: of the top 50 pages my ranking says to review first,
what fraction are actually in the "declining" proxy label? This matches how the output would
really be used — a review team has limited capacity, so what matters is whether the top of the
list is trustworthy, not overall accuracy across all 30,000 pages. In the starter notebooks,
the hand-written rule scored 0.240 on this metric, and a random forest scored 0.740 — meaning
"good" for my lane means clearing well above the ~0.24 baseline a simple rule already sets.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 4. The unit of analysis, as a real dataframe

*(see code above)* One row represents one content page (`content_id`), belonging to one
client (`client_id`), with its own visibility, staleness, and trend signals. This is the level
at which the final ranking and recommendation happen — a reviewer looks at individual pages,
not clients or dates, so the page is the correct grain for this lane.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# One row = one content page (content_id), which is the unit of analysis for this lane.
unit_of_analysis = df[["content_id", "client_id", "impressions_90d",
                        "days_since_last_update", "avg_position",
                        "trend_direction"]].head(5)

print("One row = one content page. Example rows:")
unit_of_analysis

One row = one content page. Example rows:


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,down
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,down
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML beats a fixed rule here

A fixed rule like "stale AND still visible" is easy to write but only looks at 2 signals at a
time, combined in one rigid way. Real pages vary across many signals at once — staleness,
visibility, position, CTR, word count, engagement — and how these interact isn't a clean
straight line; a page might be stale but fine because it's still ranking #1, or fresh but
struggling because of poor engagement. The Week 1/2 starter notebooks already showed this
empirically: the hand rule reached Precision@50 of 0.240, while a random forest reached 0.740
— roughly 3x better — by learning patterns across multiple signals at once that a single
if-statement can't capture. That gap is the evidence that this pattern is real but too tangled
for a fixed rule alone.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.